# Bootstrap: Grafana OTEL → Databricks secret

**Committed template (no secrets).** Copy to **`notebooks/otel/grafana/secrets-bootstrap.local.ipynb`** (gitignored), paste real values there, **or** fill the variables in a private clone and never commit that edit.

**Keep the two notebooks in sync:** `secrets-bootstrap.ipynb` and **`secrets-bootstrap.local.ipynb`** should use the same scope, secret key, **`payload`** keys, and PUT/merge logic. When you change any of that in one file, update the other so the local copy stays a drop-in mirror of this template (with real credentials only in `.local`).

This notebook writes Databricks secret scope **`lfczerobusdemo`** / key **`otel-grafana-rslee6392`**. The same JSON is merged at runtime by **`zerobus_otel_lab.py`** (used from **`zerobus-otel.ipynb`** in this folder) into **`OTEL_CONFIG`** for blank fields. To persist edits from Python, call **`zerobus_otel_lab.save_otel_grafana_to_databricks_secret(OTEL_CONFIG)`**.

Once the secret exists, you do not need to re-run this notebook for normal **`zerobus-otel.ipynb`** runs—only when **rotating** Grafana credentials or changing fields you store in the secret.

**Same Databricks scope** as **`notebooks/public_example.ipynb`**: **`lfczerobusdemo`**. Grafana OTLP uses key **`otel-grafana-rslee6392`**; ZeroBus SP/OAuth/endpoint JSON uses key **`lfczerobusdemo`** (same as `public_example`). **`zerobus-otel.ipynb`** reads both.

---

## 1. Grafana Cloud

1. Create a stack at [Grafana Cloud](https://grafana.com/products/cloud/) (free tier includes OTLP ingestion).

2. **OpenTelemetry / collector context:** [Grafana Alloy — OpenTelemetry Collector distribution](https://grafana.com/oss/alloy-opentelemetry-collector/) (optional collector in front of backends).

3. **Credentials in your Grafana stack:** **Connections → Add new connection → OpenTelemetry (collector)**  
   URL shape: `https://<your-stack>.grafana.net/connections/add-new-connection/collector-open-telemetry`  
   Copy the environment table from that page:
   - **`GRAFANA_CLOUD_OTLP_ENDPOINT`** → set **`GRAFANA_OTLP_ENDPOINT`** in the code cell below (host or `.../otlp` from Grafana, e.g. `https://otlp-gateway-prod-us-west-0.grafana.net`). **`zerobus_otel_lab`** builds OTLP/HTTP URLs: for **`*.grafana.net`** it uses **`/otlp/v1/traces`** and **`/otlp/v1/metrics`** when the path has no `/otlp`; if you store **`.../otlp`**, it appends **`/v1/traces`** / **`/v1/metrics`** only.
   - **`GRAFANA_CLOUD_BASIC_AUTH_HEADER`** → set **`GRAFANA_BASIC_AUTH_HEADER`** (the full **`Basic …`** value). *Alternatively* leave that empty and set **`GRAFANA_INSTANCE_ID`** + **`GRAFANA_API_TOKEN`** so clients can build Basic auth.

   Official reference: [Send OTLP data to Grafana Cloud](https://grafana.com/docs/grafana-cloud/send-data/otlp/send-data-otlp/).

4. **Optional — “Grafana traces:” link in `zerobus-otel`:** Set **`GRAFANA_STACK_URL`** to your Grafana **browser** URL (`https://<your-stack>.grafana.net` — not the `otlp-gateway-…` host). Set **`GRAFANA_TRACES_DATASOURCE_UID`** to the **UID** on the traces datasource **Settings** tab (often **`grafanacloud-<stack>-traces`**). **Do not** paste the **HTTP URL** field (e.g. `https://tempo-prod-….grafana.net/tempo`); that is the backend endpoint, not the Explore-link UID. Leave UID empty to open **Explore** without TraceQL pre-fill.

5. Fill the variables in the code cell below, then run **all cells**. That **PUT**s the secret.

6. Run **`notebooks/otel/grafana/zerobus-otel.ipynb`** for the ZeroBus + OTEL demo (it imports **`zerobus_otel_lab.py`**, which reads this secret when `OTEL_CONFIG` fields are blank).

View telemetry in Grafana → **Explore** or **Drilldown**.


In [ ]:
%pip install --quiet databricks-sdk

In [ ]:
# Paste from Grafana (GRAFANA_CLOUD_* in the UI). Leave token/header empty until you have real values.

GRAFANA_OTLP_ENDPOINT = ""  # e.g. https://otlp-gateway-prod-us-west-0.grafana.net
GRAFANA_BASIC_AUTH_HEADER = ""  # full "Basic …" from GRAFANA_CLOUD_BASIC_AUTH_HEADER

# Optional if you skip BASIC_AUTH_HEADER: instance id + API token (notebook / zerobus-otel can build Basic)
GRAFANA_INSTANCE_ID = ""
GRAFANA_API_TOKEN = ""

# Optional: print "Grafana traces: https://…" in zerobus-otel (browser stack URL, not OTLP gateway)
GRAFANA_STACK_URL = ""  # e.g. https://myorg.grafana.net
GRAFANA_TRACES_DATASOURCE_UID = ""  # Settings → UID (e.g. grafanacloud-<stack>-traces), NOT tempo-prod…/tempo URL

In [ ]:
import json

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceAlreadyExists

_SECRET_SCOPE = "lfczerobusdemo"
_OTEL_GRAFANA_KEY = "otel-grafana-rslee6392"

payload = {
    "GRAFANA_OTLP_ENDPOINT": GRAFANA_OTLP_ENDPOINT.strip(),
    "GRAFANA_BASIC_AUTH_HEADER": GRAFANA_BASIC_AUTH_HEADER.strip(),
    "GRAFANA_INSTANCE_ID": GRAFANA_INSTANCE_ID.strip(),
    "GRAFANA_API_TOKEN": GRAFANA_API_TOKEN.strip(),
    "GRAFANA_STACK_URL": GRAFANA_STACK_URL.strip(),
    "GRAFANA_TRACES_DATASOURCE_UID": GRAFANA_TRACES_DATASOURCE_UID.strip(),
}

if not payload["GRAFANA_BASIC_AUTH_HEADER"] and not (
    payload["GRAFANA_INSTANCE_ID"] and payload["GRAFANA_API_TOKEN"]
):
    raise ValueError(
        "Set GRAFANA_BASIC_AUTH_HEADER (recommended) or both GRAFANA_INSTANCE_ID and GRAFANA_API_TOKEN"
    )

w = WorkspaceClient()

try:
    w.secrets.create_scope(_SECRET_SCOPE)
    print(f"Created scope {_SECRET_SCOPE!r}")
except ResourceAlreadyExists:
    print(f"Scope {_SECRET_SCOPE!r} already exists")

# Merge with existing JSON if dbutils can read the key (Databricks / Connect).
current = {}
try:
    raw = dbutils.secrets.get(scope=_SECRET_SCOPE, key=_OTEL_GRAFANA_KEY)
    current = json.loads(raw)
except Exception:
    pass

current.update(payload)
w.secrets.put_secret(
    scope=_SECRET_SCOPE,
    key=_OTEL_GRAFANA_KEY,
    string_value=json.dumps(current, indent=2),
)
print(f"Updated {_SECRET_SCOPE}/{_OTEL_GRAFANA_KEY} keys: {list(payload.keys())}")